In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch

data = [
    ("الخدمة ممتازة جدا شكرا", 2),
    ("انا مبسوط جدا من التعامل", 2),
    ("منتج رائع انصح بيه", 2),
    ("شكرا على السرعة", 2),
    ("التوصيل كان سريع ومحترم", 2),
    ("الخدمة عادية مش وحشة", 1),
    ("مقبول بس ممكن يتحسن", 1),
    ("عادي يعني", 1),
    ("مش احسن حاجة بس ماشي", 1),
    ("السعر غالي شوية", 1),
    ("انا زعلان جدا الخدمة سيئة", 0),
    ("الاوردر اتأخر 3 ايام", 0),
    ("محدش بيرد عليا", 0),
    ("المنتج وصل بايظ", 0),
    ("تجربة سيئة مش هكررها", 0),
]
data = data * 10
df = pd.DataFrame(data, columns=['text', 'label'])

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

model_name = "asafaya/bert-base-arabic"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {'accuracy': accuracy_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())

config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/62.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/334k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

model.safetensors: downloading bytes:           |  0.00B            

... (4 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.677477
20,0.085258


Training Loss,Validation Loss,Step,Accuracy
0.085258,0.021638,24,1.000000


{'eval_loss': 0.021638154983520508, 'eval_accuracy': 1.0}


In [2]:
def predict_comment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    import torch
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()
    labels = {0: "سلبي 😡", 1: "محايد 😐", 2: "ايجابي 😍"}
    return labels[pred]

print(predict_comment("الخدمة وحشة جدا ومحدش بيرد"))
print(predict_comment("شكرا جدا انتو ناس محترمة"))
print(predict_comment("الاوردر اتأخر 3 ايام"))
print(predict_comment("التوصيل كان سريع"))

سلبي 😡
ايجابي 😍
سلبي 😡
ايجابي 😍
